# CaImAn results: motion correction and component visualization (from `caiman_results.hdf5`)

This notebook is a **minimal, presentation-friendly** workflow to load a CaImAn CNMF(E) results `.hdf5` file and generate figures for:

- Motion correction diagnostics (**estimated shifts** over time)
- Spatial components (footprints) visualization
- Temporal traces and deconvolved activity (if present)
- Quick component browser for screenshots for slides

## Notes (important)
- Many CaImAn pipelines **do not store the corrected movie** inside the `.hdf5`.
  From the `.hdf5` alone, we can usually visualize motion estimates (shifts) and CNMF results (A, C, etc.).
- If you also want before/after movie visualizations, you typically need the original movie (or CaImAn memmap) used during motion correction.

In [ ]:
# =====================
# USER SETTINGS
# =====================
from pathlib import Path

RESULTS_PATH = Path("/media/toor/SeagateClean/Mouse946/Round1/Linear/2026-02-05T16_54_25/caiman_final/caiman_results.hdf5")

FIG_DIR = RESULTS_PATH.parent / "figures_for_presentation"
FIG_DIR.mkdir(exist_ok=True, parents=True)

print("RESULTS_PATH:", RESULTS_PATH)
print("FIG_DIR:", FIG_DIR)
assert RESULTS_PATH.exists(), f"File not found: {RESULTS_PATH}"

## Imports

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy import sparse

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

plt.rcParams["figure.dpi"] = 120

## Helper functions (robust across CaImAn versions)

In [ ]:
def h5_list(h5obj):
    # Recursively list dataset paths
    out = []
    def _walk(obj, path=""):
        if isinstance(obj, h5py.Dataset):
            out.append(path)
        elif isinstance(obj, h5py.Group):
            for k in obj.keys():
                _walk(obj[k], f"{path}/{k}" if path else f"/{k}")
    _walk(h5obj, "")
    return out

def h5_get_first(h5, candidates, allow_none=True):
    # Return first existing dataset among candidates
    for k in candidates:
        if k in h5:
            return np.array(h5[k])
    if allow_none:
        return None
    raise KeyError(f"None of these keys were found: {candidates}")

def h5_find_contains(h5, substrings):
    # Find dataset paths containing any substring (case-insensitive)
    paths = h5_list(h5)
    subs = [s.lower() for s in substrings]
    hits = [p for p in paths if any(s in p.lower() for s in subs)]
    return sorted(hits)

def as_2d_shifts(shifts):
    # Normalize shifts to shape (T, 2) for rigid motion (dy, dx)
    if shifts is None:
        return None
    arr = np.asarray(shifts)
    if arr.ndim == 1:
        return arr.reshape(-1, 1)
    if arr.ndim == 2:
        if arr.shape[1] in (2, 3):
            return arr
        if arr.shape[0] in (2, 3):
            return arr.T
    return arr

## Inspect the `.hdf5` contents (helps if key names differ)

In [ ]:
with h5py.File(RESULTS_PATH, "r") as h5:
    paths = h5_list(h5)
    print(f"Found {len(paths)} datasets.")
    for p in paths[:80]:
        print(p)
    if len(paths) > 80:
        print("...")

## Load CaImAn outputs (A, C, S, dims, shifts)

In [ ]:
def load_caiman_results(results_path):
    out = {}
    with h5py.File(results_path, "r") as h5:
        # dims
        d1 = h5_get_first(h5, ["/d1", "/dims/d1", "/cnmf/d1", "/params/d1"])
        d2 = h5_get_first(h5, ["/d2", "/dims/d2", "/cnmf/d2", "/params/d2"])
        dims = h5_get_first(h5, ["/dims", "/cnmf/dims", "/params/dims"])
        if dims is not None and np.size(dims) >= 2:
            dims = tuple(int(x) for x in np.ravel(dims)[:2])
        elif d1 is not None and d2 is not None:
            dims = (int(np.ravel(d1)[0]), int(np.ravel(d2)[0]))
        else:
            dims = None
        out["dims"] = dims

        # temporal
        out["C"] = h5_get_first(h5, ["/C", "/cnmf/C", "/estimates/C", "/cnmfe/C"])
        out["S"] = h5_get_first(h5, ["/S", "/cnmf/S", "/estimates/S", "/cnmfe/S"])
        out["F_dff"] = h5_get_first(h5, ["/F_dff", "/cnmf/F_dff", "/estimates/F_dff"])

        # background (optional)
        out["b"] = h5_get_first(h5, ["/b", "/cnmf/b", "/estimates/b"])
        out["f"] = h5_get_first(h5, ["/f", "/cnmf/f", "/estimates/f"])

        # spatial A (dense or sparse)
        A_dense = h5_get_first(h5, ["/A", "/cnmf/A", "/estimates/A", "/cnmfe/A"])
        if A_dense is not None:
            out["A"] = A_dense
        else:
            out["A"] = None
            for base in ["/A", "/cnmf/A", "/estimates/A", "/cnmfe/A"]:
                if f"{base}/data" in h5 and f"{base}/indices" in h5 and f"{base}/indptr" in h5 and f"{base}/shape" in h5:
                    data = np.array(h5[f"{base}/data"])
                    indices = np.array(h5[f"{base}/indices"])
                    indptr = np.array(h5[f"{base}/indptr"])
                    shape = tuple(int(x) for x in np.array(h5[f"{base}/shape"]).ravel())
                    out["A"] = sparse.csc_matrix((data, indices, indptr), shape=shape)
                    break

        # summary images (optional)
        out["Cn"] = h5_get_first(h5, ["/Cn", "/cnmf/Cn", "/estimates/Cn", "/correlation_image", "/cn_filter"])
        out["mean_img"] = h5_get_first(h5, ["/mean", "/mean_img", "/cnmf/mean", "/estimates/mean"])

        # motion (optional, key names vary)
        out["shifts_rig"] = h5_get_first(h5, [
            "/shifts_rig", "/motion/shifts_rig", "/moco/shifts_rig", "/motion_correction/shifts_rig",
            "/shifts", "/motion/shifts"
        ])
        out["shifts_els"] = h5_get_first(h5, [
            "/shifts_els", "/motion/shifts_els", "/moco/shifts_els", "/motion_correction/shifts_els",
            "/shifts_pwrigid", "/motion/shifts_pwrigid"
        ])
        out["tmpl"] = h5_get_first(h5, ["/template", "/motion/template", "/moco/template", "/motion_correction/template", "/tmpl"])

        out["_debug_motion_hits"] = h5_find_contains(h5, ["shift", "motion", "moco", "template"])
    return out

res = load_caiman_results(RESULTS_PATH)

print("dims:", res["dims"])
for k in ["A", "C", "S", "F_dff", "b", "f", "Cn", "mean_img", "shifts_rig", "shifts_els", "tmpl"]:
    v = res.get(k, None)
    if v is None:
        print(f"{k}: None")
    else:
        if sparse.issparse(v):
            print(f"{k}: sparse {v.shape}, nnz={v.nnz}")
        else:
            arr = np.asarray(v)
            print(f"{k}: {arr.shape}, dtype={arr.dtype}")

print("\nMotion-related dataset hits (debug):")
for p in res["_debug_motion_hits"][:40]:
    print(" ", p)

## Motion correction diagnostics (from shifts)

In [ ]:
def plot_shifts(shifts, title, save_name=None):
    if shifts is None:
        print(f"[skip] {title}: shifts not found in hdf5.")
        return
    sh = as_2d_shifts(shifts)

    plt.figure(figsize=(10, 3))
    if sh.shape[1] >= 2:
        plt.plot(sh[:, 0], label="dy")
        plt.plot(sh[:, 1], label="dx")
    else:
        plt.plot(sh[:, 0], label="shift")
    plt.ylabel("pixels")
    plt.xlabel("frame")
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    if save_name:
        out = FIG_DIR / save_name
        plt.savefig(out, bbox_inches="tight")
        print("Saved:", out)

    plt.figure(figsize=(6, 3))
    if sh.shape[1] >= 2:
        mag = np.sqrt(sh[:, 0] ** 2 + sh[:, 1] ** 2)
        plt.hist(mag, bins=60)
        plt.xlabel("shift magnitude (pixels)")
    else:
        plt.hist(sh[:, 0], bins=60)
        plt.xlabel("shift (pixels)")
    plt.ylabel("count")
    plt.title(f"{title} (distribution)")
    plt.tight_layout()
    if save_name:
        out = FIG_DIR / save_name.replace(".png", "_hist.png")
        plt.savefig(out, bbox_inches="tight")
        print("Saved:", out)

plot_shifts(res.get("shifts_rig"), "Rigid motion correction shifts", "motion_shifts_rigid.png")
plot_shifts(res.get("shifts_els"), "Piecewise-rigid motion correction shifts", "motion_shifts_pwrigid.png")

## Template image used for motion correction (if saved)

In [ ]:
tmpl = res.get("tmpl", None)
if tmpl is None:
    print("[skip] No template image found in hdf5.")
else:
    img = np.asarray(tmpl).squeeze()
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title("Motion correction template")
    plt.axis("off")
    plt.tight_layout()
    out = FIG_DIR / "motion_template.png"
    plt.savefig(out, bbox_inches="tight")
    print("Saved:", out)

## Spatial components (A)

In [ ]:
A = res.get("A", None)
dims = res.get("dims", None)

if A is None:
    raise RuntimeError("Could not find spatial matrix A in this .hdf5. Check the printed dataset list.")
if dims is None:
    raise RuntimeError("Could not infer image dims (d1, d2). Set dims manually here if needed.")

d1, d2 = dims
n_pix = d1 * d2

# A is usually (pixels x K)
if sparse.issparse(A):
    if A.shape[0] != n_pix and A.shape[1] == n_pix:
        A = A.T
else:
    A = np.asarray(A)
    if A.shape[0] != n_pix and A.shape[1] == n_pix:
        A = A.T

K = A.shape[1]
print(f"A shape: {A.shape} (pixels x K), K={K}, dims={dims}")

In [ ]:
def get_component_image(A, k, d1, d2, normalize=True):
    if sparse.issparse(A):
        vec = A[:, k].toarray().ravel()
    else:
        vec = np.asarray(A[:, k]).ravel()
    img = vec.reshape(d1, d2, order="F")  # Fortran order is typical for CaImAn
    if normalize:
        vmin, vmax = np.percentile(img, 5), np.percentile(img, 99.5)
        if vmax > vmin:
            img = (img - vmin) / (vmax - vmin)
            img = np.clip(img, 0, 1)
    return img

def plot_component_grid(A, d1, d2, ks, ncols=6, title=None, save_name=None):
    n = len(ks)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    plt.figure(figsize=(2.2*ncols, 2.2*nrows))
    for i, k in enumerate(ks):
        ax = plt.subplot(nrows, ncols, i+1)
        ax.imshow(get_component_image(A, k, d1, d2), cmap="gray")
        ax.set_title(f"k={k}", fontsize=9)
        ax.axis("off")
    if title:
        plt.suptitle(title, y=1.02)
    plt.tight_layout()
    if save_name:
        out = FIG_DIR / save_name
        plt.savefig(out, bbox_inches="tight")
        print("Saved:", out)

plot_component_grid(A, d1, d2, ks=list(range(min(K, 24))),
                    title="Example spatial components (footprints)",
                    save_name="components_grid_first24.png")

### Max projection across all components (overview figure)

In [ ]:
if sparse.issparse(A):
    block = 256
    max_vec = np.zeros((n_pix,), dtype=np.float32)
    for start in range(0, K, block):
        end = min(K, start + block)
        max_vec = np.maximum(max_vec, np.asarray(A[:, start:end].max(axis=1)).ravel())
else:
    max_vec = np.max(A, axis=1).astype(np.float32)

max_img = max_vec.reshape(d1, d2, order="F")
plt.figure(figsize=(6, 5))
plt.imshow(max_img, cmap="gray")
plt.title("Max projection across all spatial components")
plt.axis("off")
plt.tight_layout()
out = FIG_DIR / "components_max_projection.png"
plt.savefig(out, bbox_inches="tight")
print("Saved:", out)

## Temporal traces (C) and deconvolved activity (S)

In [ ]:
C = res.get("C", None)
S = res.get("S", None)

if C is None:
    print("[warn] Temporal matrix C not found.")
else:
    C = np.asarray(C)
    if C.shape[0] != K and C.shape[1] == K:
        C = C.T
    print("C shape:", C.shape)

if S is not None:
    S = np.asarray(S)
    if S.shape[0] != K and S.shape[1] == K:
        S = S.T
    print("S shape:", S.shape)

In [ ]:
def plot_traces(C, S=None, ks=(0,1,2,3,4), tlim=None, save_name=None):
    ks = list(ks)
    T = C.shape[1]
    if tlim is None:
        t0, t1 = 0, T
    else:
        t0, t1 = tlim
    tt = np.arange(t0, t1)

    plt.figure(figsize=(10, 5))
    offset = 0.0
    for k in ks:
        y = C[k, t0:t1]
        y = (y - np.median(y)) / (np.std(y) + 1e-8)
        plt.plot(tt, y + offset, lw=1, label=f"C k={k}")
        if S is not None:
            s = S[k, t0:t1]
            s = s / (np.percentile(s, 99) + 1e-8)
            plt.plot(tt, s + offset, lw=0.8, alpha=0.7, label=f"S k={k}")
        offset += 4.0

    plt.xlabel("frame")
    plt.ylabel("z-scored (stacked)")
    plt.title("Example temporal traces (C) and deconvolved activity (S)")
    plt.legend(ncol=2, fontsize=8, frameon=False, loc="upper right")
    plt.tight_layout()

    if save_name:
        out = FIG_DIR / save_name
        plt.savefig(out, bbox_inches="tight")
        print("Saved:", out)

if C is not None:
    plot_traces(C, S=S if S is not None else None, ks=range(min(8, K)), save_name="example_traces.png")

## Interactive component browser (optional)

In [ ]:
if not HAS_WIDGETS:
    print("ipywidgets not available. Use the static plots above.")
else:
    def show_component(k):
        img = get_component_image(A, k, d1, d2)
        plt.figure(figsize=(5, 5))
        plt.imshow(img, cmap="gray")
        plt.title(f"Component k={k}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        if C is not None:
            plt.figure(figsize=(10, 2.5))
            plt.plot(C[k], lw=1)
            plt.title(f"Trace C for k={k}")
            plt.xlabel("frame")
            plt.tight_layout()
            plt.show()

        if S is not None:
            plt.figure(figsize=(10, 2.5))
            plt.plot(S[k], lw=1)
            plt.title(f"Deconvolved activity S for k={k}")
            plt.xlabel("frame")
            plt.tight_layout()
            plt.show()

    slider = widgets.IntSlider(value=0, min=0, max=K-1, step=1, description="k")
    display(widgets.HBox([slider]), widgets.interactive_output(show_component, {"k": slider}))

## Optional images: correlation image (Cn) or mean image (if present)

In [ ]:
for key, name in [("Cn", "Correlation image (Cn)"), ("mean_img", "Mean image")]:
    img = res.get(key, None)
    if img is None:
        continue
    im = np.asarray(img).squeeze()
    plt.figure(figsize=(6, 5))
    plt.imshow(im, cmap="gray")
    plt.title(name)
    plt.axis("off")
    plt.tight_layout()
    out = FIG_DIR / f"{key}.png"
    plt.savefig(out, bbox_inches="tight")
    print("Saved:", out)

## Presentation-friendly interpretation
- Motion correction: show shift traces (dx/dy over frames) and template image (if available).
- CNMF decomposition: show spatial footprints (A) and temporal activity (C), plus deconvolved activity (S) if present.
- Overview: component grid and max projection.

If you want before vs after movies, you need the raw movie or memmap, not only the `.hdf5`.